# Training Notebook (Spark ML, Distributed)

Trains Spark ML baselines on pre-engineered features and saves outputs to both HDFS and local mounted folders.

Main outputs:
- HDFS metrics/predictions by run id


In [1]:
# Cần cài trên cả master và các worker
%pip install scikit-learn pyarrow statsmodels xgboost


Note: you may need to restart the kernel to use updated packages.


In [2]:
import json
import os
import urllib.request
import pandas as pd

from pyspark.sql import SparkSession, functions as F
from pyspark.ml import Pipeline
from pyspark.ml.feature import VectorAssembler
from pyspark.ml.regression import LinearRegression, RandomForestRegressor
from xgboost.spark import SparkXGBRegressor
from pyspark.ml.evaluation import RegressionEvaluator

spark = (
    SparkSession.builder
    .appName("DemandPredictionTrainingSparkML_VirtualCluster")
    .master("spark://master:7077")
    .config("spark.eventLog.enabled", "true")
    .config("spark.eventLog.dir", "hdfs://master:9000/spark-logs")
    .config("spark.driver.memory", "1500m")
    .config("spark.driver.memoryOverhead", "512m")
    .config("spark.executor.memory", "2g")
    .config("spark.executor.memoryOverhead", "512m")
    .config("spark.executor.cores", "2")
    .config("spark.cores.max", "6")
    .config("spark.sql.shuffle.partitions", "24")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")

def cluster_util(stage_name):
    import json as _json, urllib.request
    print(f"===== CLUSTER UTILIZATION: {stage_name} =====")
    print("Spark master:", spark.sparkContext.master)
    try:
        info = _json.load(urllib.request.urlopen("http://master:8080/json/"))
        workers = info.get("workers", [])
        alive = [w for w in workers if w.get("state") == "ALIVE"]
        apps = info.get("activeapps", [])
        print("alive workers:", len(alive))
        print("active apps :", len(apps))
        for w in alive:
            print("worker", w.get("id"),
                  "cores", f"{w.get('coresused',0)}/{w.get('cores',0)}",
                  "mem(MB)", f"{w.get('memoryused',0)}/{w.get('memory',0)}")
    except Exception as e:
        print("Could not query Spark Standalone Master:", e)

cluster_util("session_started")


:: loading settings :: url = jar:file:/opt/spark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /root/.ivy2/cache
The jars for the packages stored in: /root/.ivy2/jars
graphframes#graphframes added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-ff4f48ee-cbcc-42f8-89e0-b2f7eaa8b3de;1.0
	confs: [default]
	found graphframes#graphframes;0.8.3-spark3.5-s_2.12 in spark-packages
	found org.slf4j#slf4j-api;1.7.16 in central
:: resolution report :: resolve 88ms :: artifacts dl 4ms
	:: modules in use:
	graphframes#graphframes;0.8.3-spark3.5-s_2.12 from spark-packages in [default]
	org.slf4j#slf4j-api;1.7.16 from central in [default]
	---------------------------------------------------------------------
	|                  |            modules            ||   artifacts   |
	|       conf       | number| search|dwnlded|evicted|| number|dwnlded|
	---------------------------------------------------------------------
	|      default     |   2   |   0   |   0   |   0   ||   2   |   0   |
	------------------------------------------------

===== CLUSTER UTILIZATION: session_started =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048


In [3]:
FEATURE_PATH = "/user/data/feature_engineering/demand_prediction_features"
OUT_BASE_ROOT = "/user/data/results/demand_prediction/sparkml"
TARGET_COL = "pickup_demand"

TRAIN_START_TS = None
TRAIN_END_TS = None

feature_cols = [
    "hour", "dow", "month", "is_weekend",
    "lag_1", "lag_2", "lag_3", "lag_6", "lag_12", "lag_24",
    "roll_mean_6", "roll_mean_18", "roll_std_18",
]

df_all = spark.read.parquet(FEATURE_PATH)

if "split_local" not in df_all.columns and "split" in df_all.columns:
    # Backward compatibility with feature_engineering output
    df_all = df_all.withColumn("split_local", F.col("split"))
elif "split_local" not in df_all.columns:
    raise ValueError("Missing split column: expected split_local or split in feature dataset.")

# Tùy thuộc vào bước FE bạn dùng tên cột nào (10m hay 60m)
TIME_COL = "pickup_bin_10m" if "pickup_bin_10m" in df_all.columns else "pickup_bin_60m"
print(f"Using time column: {TIME_COL}")

available_bounds = df_all.agg(
    F.min(TIME_COL).alias("min_ts"),
    F.max(TIME_COL).alias("max_ts")
).first()

effective_start = TRAIN_START_TS if TRAIN_START_TS is not None else available_bounds["min_ts"]
effective_end = TRAIN_END_TS if TRAIN_END_TS is not None else available_bounds["max_ts"]

df = df_all.where((F.col(TIME_COL) >= effective_start) & (F.col(TIME_COL) <= effective_end)).cache()
print(f"Rows in window: {df.count()}")

Using time column: pickup_bin_10m


Rows in window: 12041736


In [4]:
# --- HYBRID APPROACH: HOLT-WINTERS FEATURE ---
import pandas as pd
import numpy as np
from pyspark.sql.types import StructType, StructField, DoubleType

# Use DAILY seasonality (144 bins = 24h * 6 bins/h).
# Weekly (1008) is too expensive: each zone has ~185K rows and
# ExponentialSmoothing needs to estimate 1008 initial seasonal
# components, causing executor OOM on a 2GB-per-executor cluster.
HW_SEASONAL_PERIODS = 24  # 1 day (24h x 6 ten-minute bins)
HW_MIN_ROWS = HW_SEASONAL_PERIODS * 3  # at least 3 days of history
HW_MIN_TRAIN = HW_SEASONAL_PERIODS * 2  # at least 2 days in train set
# Cap training input to the most recent data to control memory/CPU per zone
HW_MAX_TRAIN = HW_SEASONAL_PERIODS * 52  # ~1 year of daily patterns

def add_hw_forecast(pdf: pd.DataFrame) -> pd.DataFrame:
    pdf = pdf.sort_values("pickup_bin_10m").reset_index(drop=True)
    y = pdf["pickup_demand"].astype(float).to_numpy()

    # Not enough data - fall back to actual demand
    if len(y) < HW_MIN_ROWS:
        pdf["hw_forecast"] = y
        return pdf

    train_mask = pdf["split_local"] == "train"
    y_train_full = y[train_mask.to_numpy()]

    if len(y_train_full) < HW_MIN_TRAIN:
        pdf["hw_forecast"] = y
        return pdf

    try:
        from statsmodels.tsa.holtwinters import ExponentialSmoothing

        # Use only the most recent HW_MAX_TRAIN points to bound memory+CPU
        y_fit = y_train_full[-HW_MAX_TRAIN:]
        n_fit = len(y_fit)
        n_train = len(y_train_full)
        n_test = int((~train_mask).sum())

        hw = ExponentialSmoothing(
            y_fit,
            trend="add",
            seasonal="add",
            seasonal_periods=HW_SEASONAL_PERIODS,
            initialization_method="estimated",
        )
        fitted = hw.fit(optimized=True)
        in_sample = np.asarray(fitted.fittedvalues)  # shape (n_fit,)

        # Build full training forecast (older rows not in fit window use first fitted value)
        n_older = n_train - n_fit
        if n_older > 0:
            train_forecast = np.concatenate([
                np.full(n_older, in_sample[0]),
                in_sample
            ])
        else:
            train_forecast = in_sample

        # Forecast test data
        if n_test > 0:
            test_forecast = np.asarray(fitted.forecast(n_test))
            forecast_all = np.concatenate([train_forecast, test_forecast])
        else:
            forecast_all = train_forecast

        pdf["hw_forecast"] = np.maximum(forecast_all, 0.0)

    except Exception:
        pdf["hw_forecast"] = y

    return pdf

print(f"Applying distributed Holt-Winters (seasonal_periods={HW_SEASONAL_PERIODS}, max_train={HW_MAX_TRAIN}) to all zones...")
hw_schema = StructType(df.schema.fields + [StructField("hw_forecast", DoubleType(), True)])
df = df.groupBy("PULocationID").applyInPandas(add_hw_forecast, schema=hw_schema).cache()

if "hw_forecast" not in feature_cols:
    feature_cols.append("hw_forecast")

train_df = df.where(F.col("split_local") == "train")
test_df = df.where(F.col("split_local") == "test")
print("Hybrid Train rows:", train_df.count())
print("Hybrid Test rows :", test_df.count())
cluster_util("after_hw_feature")


Applying distributed Holt-Winters (seasonal_periods=24, max_train=1248) to all zones...


Hybrid Train rows: 8429032
Hybrid Test rows : 3612704
===== CLUSTER UTILIZATION: after_hw_feature =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048


In [5]:
assembler = VectorAssembler(inputCols=feature_cols, outputCol="features", handleInvalid="skip")

train_fit_df = train_df
mae_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="mae")
rmse_eval = RegressionEvaluator(labelCol=TARGET_COL, predictionCol="prediction", metricName="rmse")

metrics_rows = []
pred_union = None
fitted_models = {}

def collect_metrics_and_predictions(model_name, pred_df):
    global pred_union
    mae_val = float(mae_eval.evaluate(pred_df))
    rmse_val = float(rmse_eval.evaluate(pred_df))
    mape_val = (
        pred_df.where(F.col(TARGET_COL) > 0)
        .select((F.abs(F.col(TARGET_COL) - F.col("prediction")) / F.col(TARGET_COL)).alias("ape"))
        .agg((F.avg(F.col("ape")) * 100.0).alias("mape"))
        .first()["mape"]
    )

    metrics_rows.append({
        "model": model_name,
        "MAE": mae_val,
        "RMSE": rmse_val,
        "MAPE": float(mape_val) if mape_val is not None else None,
    })

    slim_pred = pred_df.select(
        F.lit(model_name).alias("model"),
        "PULocationID",
        "pickup_bin_10m",
        TARGET_COL,
        "prediction",
    )
    pred_union = slim_pred if pred_union is None else pred_union.unionByName(slim_pred)


In [6]:
# Model 1: Linear Regression
model_name = "linear_regression"
cluster_util(f"before_train_{model_name}")

lr = LinearRegression(
    featuresCol="features",
    labelCol=TARGET_COL,
    predictionCol="prediction",
    maxIter=50,            # Was 100: L-BFGS iterations on 57M rows is heavy
    regParam=0.1,
    elasticNetParam=0.0,
    aggregationDepth=3,    # Tree-reduce instead of all-reduce: reduces per-executor memory spikes
)

lr_pipeline = Pipeline(stages=[assembler, lr])
lr_fitted = lr_pipeline.fit(train_fit_df)
fitted_models[model_name] = lr_fitted

lr_pred = lr_fitted.transform(test_df).withColumn(
    "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
)
collect_metrics_and_predictions(model_name, lr_pred)
cluster_util(f"after_train_{model_name}")
print("Finished:", model_name)


===== CLUSTER UTILIZATION: before_train_linear_regression =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048


26/04/29 08:17:28 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.blas.JNIBLAS
26/04/29 08:17:28 WARN InstanceBuilder: Failed to load implementation from:dev.ludovic.netlib.lapack.JNILAPACK
26/04/29 08:17:34 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


===== CLUSTER UTILIZATION: after_train_linear_regression =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048
Finished: linear_regression


In [7]:
# Model 2: Random Forest
model_name = "random_forest"
cluster_util(f"before_train_{model_name}")

rf = RandomForestRegressor(
    featuresCol="features",
    labelCol=TARGET_COL,
    predictionCol="prediction",
    numTrees=30,   # Conservative for 2GB executor: 50 trees at depth 8 will OOM
    maxDepth=6,
    maxBins=32,
    seed=42,
)

rf_pipeline = Pipeline(stages=[assembler, rf])
rf_fitted = rf_pipeline.fit(train_fit_df)
fitted_models[model_name] = rf_fitted

rf_pred = rf_fitted.transform(test_df).withColumn(
    "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
)
collect_metrics_and_predictions(model_name, rf_pred)
cluster_util(f"after_train_{model_name}")
print("Finished:", model_name)


===== CLUSTER UTILIZATION: before_train_random_forest =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048


===== CLUSTER UTILIZATION: after_train_random_forest =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048
Finished: random_forest


In [8]:
# Model 3: XGBoost
model_name = "xgboost"
cluster_util(f"before_train_{model_name}")

xgb = SparkXGBRegressor(
    features_col="features",
    label_col=TARGET_COL,
    prediction_col="prediction",
    n_estimators=40,
    max_depth=5,
    learning_rate=0.05,
    subsample=0.7,
    num_workers=3,  # 3 Spark workers in the cluster
)

xgb_pipeline = Pipeline(stages=[assembler, xgb])
xgb_fitted = xgb_pipeline.fit(train_fit_df)
fitted_models[model_name] = xgb_fitted

xgb_pred = xgb_fitted.transform(test_df).withColumn(
    "prediction", F.when(F.col("prediction") < 0, F.lit(0.0)).otherwise(F.col("prediction"))
)
collect_metrics_and_predictions(model_name, xgb_pred)
cluster_util(f"after_train_{model_name}")
print("Finished:", model_name)


===== CLUSTER UTILIZATION: before_train_xgboost =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048


2026-04-29 08:18:34,342 INFO XGBoost-PySpark: _fit Running xgboost-3.2.0 on 3 workers with
	booster params: {'objective': 'reg:squarederror', 'device': 'cpu', 'learning_rate': 0.05, 'max_depth': 5, 'subsample': 0.7, 'nthread': 1}
	train_call_kwargs_params: {'verbose_eval': True, 'num_boost_round': 40}
	dmatrix_kwargs: {'nthread': 1, 'missing': nan}
[08:19:16] [0]	training-rmse:9.95322                                (0 + 3) / 3]
[08:19:16] [1]	training-rmse:9.91009
[08:19:17] [2]	training-rmse:9.87053
[08:19:17] [3]	training-rmse:9.83467
[08:19:17] [4]	training-rmse:9.80209
[08:19:17] [5]	training-rmse:9.77241
[08:19:17] [6]	training-rmse:9.74533
[08:19:18] [7]	training-rmse:9.72078
[08:19:18] [8]	training-rmse:9.69844
[08:19:18] [9]	training-rmse:9.67789
[08:19:18] [10]	training-rmse:9.65942
[08:19:18] [11]	training-rmse:9.64205
[08:19:19] [12]	training-rmse:9.62654
[08:19:19] [13]	training-rmse:9.61210
[08:19:19] [14]	training-rmse:9.59921
[08:19:19] [15]	training-rmse:9.58739
[08:19:

===== CLUSTER UTILIZATION: after_train_xgboost =====
Spark master: spark://master:7077
alive workers: 3
active apps : 1
worker worker-20260429081424-172.18.0.4-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081516-172.18.0.2-7078 cores 2/2 mem(MB) 2048/2048
worker worker-20260429081424-172.18.0.3-7078 cores 2/2 mem(MB) 2048/2048
Finished: xgboost


In [9]:
metrics_pdf = pd.DataFrame(metrics_rows).sort_values("MAPE")
display(metrics_pdf)

if len(metrics_pdf) == 0:
    raise ValueError("No model finished successfully. Check Spark logs/resources.")

best_model_name = metrics_pdf.iloc[0]["model"]
print("Best model by MAPE:", best_model_name)


,model,MAE,RMSE,MAPE
2,xgboost,3.301928,10.260137,69.302225
0,linear_regression,3.543409,10.491953,71.898850
1,random_forest,3.318501,10.279707,74.959302


Best model by MAPE: xgboost


In [11]:
# Ensure output base is defined for this cell-only rerun
from datetime import datetime
try:
    run_id
except NameError:
    run_id = datetime.utcnow().strftime("%Y%m%d_%H%M%S")
OUT_BASE = f"{OUT_BASE_ROOT}/run_{run_id}"

metrics_sdf = spark.createDataFrame(metrics_rows)
metrics_sdf.write.mode("overwrite").parquet(f"{OUT_BASE}/metrics")
pred_union.write.mode("overwrite").partitionBy("model").parquet(f"{OUT_BASE}/predictions")

best_model_path = f"{OUT_BASE}/best_model_pipeline"
fitted_models[best_model_name].write().overwrite().save(best_model_path)


best_pred_pdf = (
    pred_union.where(F.col("model") == best_model_name)
    .orderBy("pickup_bin_10m", "PULocationID")
    .limit(5000)
    .toPandas()
  )

run_meta = {
    "run_id": run_id,
    "feature_path": FEATURE_PATH,
    "effective_start": str(effective_start),
    "effective_end": str(effective_end),
    "hdfs_output_base": OUT_BASE,
    "best_model": best_model_name,
}
import json
print("Run Metadata:")
print(json.dumps(run_meta, indent=2))

print("Saved HDFS outputs:")
print(f"- {OUT_BASE}/metrics")
print(f"- {OUT_BASE}/predictions")

Run Metadata:
{
  "run_id": "20260429_082204",
  "feature_path": "/user/data/feature_engineering/demand_prediction_features",
  "effective_start": "2020-01-02 00:00:00",
  "effective_end": "2025-12-31 23:00:00",
  "hdfs_output_base": "/user/data/results/demand_prediction/sparkml/run_20260429_082204",
  "best_model": "xgboost"
}
Saved HDFS outputs:
- /user/data/results/demand_prediction/sparkml/run_20260429_082204/metrics
- /user/data/results/demand_prediction/sparkml/run_20260429_082204/predictions


In [12]:
# Optional cleanup: run this when training is done.
spark.catalog.clearCache()
spark.stop()
print("Training Spark session stopped.")


Training Spark session stopped.
